In [1]:
class Vertice:
    def __init__(self, rotulo):
        self.rotulo = rotulo
        self.visitado = False
        self.adjacentes = []
        self.custo_acumulado = float('inf')  # g(n)
        self.anterior = None                 # para reconstruir o caminho

    def adiciona_adjacente(self, adjacente):
        self.adjacentes.append(adjacente)

    def __repr__(self):
        return f"Vertice({self.rotulo})"

class Adjacente:
    def __init__(self, vertice, custo):
        self.vertice = vertice
        self.custo = custo

    def __repr__(self):
        return f"Adj({self.vertice.rotulo}, w={self.custo})"

class VetorOrdenado: #fila de prioridade simples: guarda tuplas (custo, vertice) ordenadas; insere mantém a ordem e pop_min tira o menor
    def __init__(self):
        self.valores = []

    def insere(self, vertice):
        # insere mantendo ordenação crescente por custo
        pos = 0
        while pos < len(self.valores) and self.valores[pos][0] <= vertice.custo_acumulado:
            pos += 1
        self.valores.insert(pos, (vertice.custo_acumulado, vertice))

    def pop_min(self):
        if not self.valores:
            return None
        return self.valores.pop(0)[1]  # retorna vertice

    def __bool__(self): #retorna True se self.valores não estiver vazio, retorna False se self.valores estiver vazio, isso permite 'while fronteira'
        return bool(self.valores)

class Grafo:
    def __init__(self):
        zurique = Vertice("Zurique")
        roma = Vertice("Roma")
        paris = Vertice("Paris")
        berlin = Vertice("Berlin")
        londres = Vertice("Londres")
        viena = Vertice("Viena")
        amsterdam = Vertice("Amsterdam")
        madrid = Vertice("Madrid")
        sofia = Vertice("Sofia")

        # lista de vértices
        self.v = [zurique, roma, paris, berlin, londres, viena, amsterdam, madrid, sofia]

        # arestas (bidirecionais) com os pesos do gráfico
        self.conecta(zurique, roma, 4)
        self.conecta(zurique, paris, 8)

        self.conecta(roma, berlin, 8)
        self.conecta(roma, paris, 11)

        self.conecta(berlin, amsterdam, 7)
        self.conecta(berlin, londres, 2)
        self.conecta(berlin, madrid, 4)

        self.conecta(londres, paris, 7)
        self.conecta(londres, viena, 6)

        self.conecta(paris, viena, 1)
        self.conecta(viena, madrid, 2)

        self.conecta(amsterdam, madrid, 14)
        self.conecta(amsterdam, sofia, 9)

        self.conecta(madrid, sofia, 10)

    def conecta(self, a, b, custo):
        a.adiciona_adjacente(Adjacente(b, custo))
        b.adiciona_adjacente(Adjacente(a, custo))

class Dijkstra:
    def __init__(self, objetivo=None):
        self.objetivo = objetivo

    def buscar(self, origem,vertices):
        # inicializa custos
        for v in vertices:
            v.visitado = False
            v.custo_acumulado = float('inf')
            v.anterior = None
        origem.custo_acumulado = 0.0

        fronteira = VetorOrdenado()
        fronteira.insere(origem)

        while fronteira:   #laço principal
            atual = fronteira.pop_min()
            if atual.visitado:
                continue

            atual.visitado = True

            if self.objetivo is not None and atual == self.objetivo:
                break
#relaxamento das arestas
            for adj in atual.adjacentes:
                v = adj.vertice
                w = adj.custo
                if w < 0:
                    raise ValueError("Dijkstra requer pesos não negativos.")
                novo_custo = atual.custo_acumulado + w
                if novo_custo < v.custo_acumulado:
                    v.custo_acumulado = novo_custo
                    v.anterior = atual
                    fronteira.insere(v)

    # ---------- reconstrói caminho genérico ----------
    def caminho_ate(self, destino):
        if destino.custo_acumulado == float('inf'):
            return None
        caminho = []
        cur = destino
        while cur is not None:
            caminho.append(cur.rotulo)
            cur = cur.anterior
        caminho.reverse()
        return caminho

if __name__ == "__main__":
    grafo = Grafo()

    # origem fixa: Zurique
    origem = next(v for v in grafo.v if v.rotulo == "Zurique")

    # mostra opções exatamente como estão no grafo
    print("Cidades disponíveis:", ", ".join(v.rotulo for v in grafo.v))
    destino_nome = input("Digite o destino exatamente como no grafo: ").strip()

    # busca EXATA pelo rótulo
    destino = None
    for v in grafo.v:
        if v.rotulo == destino_nome:
            destino = v
            break

    if destino is None:
        print("Destino inválido. Use exatamente um dos nomes mostrados.")
    elif destino is origem:
        print("O destino é a própria origem (Zurique). Custo 0. Caminho: Zurique")
    else:
        dj = Dijkstra(objetivo=destino)
        dj.buscar(origem, grafo.v)
        caminho = dj.caminho_ate(destino)
        if caminho is None:
            print(f"{origem.rotulo} -> {destino.rotulo}: sem caminho")
        else:
            print(f"{origem.rotulo} -> {destino.rotulo}: custo {destino.custo_acumulado} | caminho: {' -> '.join(caminho)}")

Cidades disponíveis: Zurique, Roma, Paris, Berlin, Londres, Viena, Amsterdam, Madrid, Sofia
Digite o destino exatamente como no grafo: Berlin
Zurique -> Berlin: custo 12.0 | caminho: Zurique -> Roma -> Berlin
